### Experimental Setup: Simulated Markov Process from Kernel $P$
We simulate a first-order Markov process in $\mathbb{R}^{d}$, where the transition distribution is a linear Gaussian:

$$
X_{n+1} \mid X_n = x \;\sim\; \mathcal{N}(A x + b, \Sigma)
$$


This defines a **contracting linear Gaussian** transition kernel with known stationary distribution.



### Closed-form Conditional Score

Because the conditional distribution is Gaussian, the conditional log density has a known closed-form gradient:

$$
\nabla_y \log p(y \mid x) = -\Sigma^{-1} (y - A x - b)
$$

This gives us the exact ground-truth **score function** for every transition pair $(x, y)$.


In [3]:
from tqdm.notebook import tqdm
import functools
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pdb

import functools
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import tqdm
from tqdm import tqdm
import matplotlib.pyplot as plt
import random

from torch.utils.data import TensorDataset, DataLoader
from Messy_data_Training_conditional_score import ConditionalScoreNet, sample_markov_chain, hyvarinen_loss, evaluate_score_convergence

from torchvision import transforms
from torch.utils.data import ConcatDataset
from torch.utils.data import Subset
from scipy import integrate
from torchvision.utils import make_grid

# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
def compute_hyvarinen_score(model, x, y):
    """
    Compute Hyvärinen score S_H(y,x;θ)
    """
    model.eval()
    x = x.to(device)
    y = y.clone().detach().to(device).requires_grad_(True)

    with torch.set_grad_enabled(True):   # allow gradients for y
        psi = model(x, y)
        norm_term = 0.5 * (psi ** 2).sum(dim=1)

        grads = []
        for i in range(psi.shape[1]):
            grad_i = torch.autograd.grad(
                psi[:, i].sum(), y, create_graph=False, retain_graph=True
            )[0][:, i]
            grads.append(grad_i)
        divergence = torch.stack(grads, dim=1).sum(dim=1)

    return norm_term + divergence


In [7]:
d = 10
hidden_dim = 128
num_layers = 3
path_path="Gaussian_change detection/hybrid_path_P0toP1.pt"
#path_path = "Gaussian_change detection/P_1_messy_markov_path_A_0.66_b_1.90_Sigma_0.09.pt"
kernel_path="Gaussian_change detection/P_1_messy_kernel_params_A_0.66_b_1.90_Sigma_0.09.pt"
# kernel_path="change detection/P_0_messy_kernel_params_A_0.73_b_1.90_Sigma_0.05.pt"

model_path="Gaussian_change detection/P_1_128_4_messy_markov_model_A_0.66_b_1.90_Sigma_0.09.pth"
# model_path="Gaussian_change detection/P_0_128_4_messy_markov_model_A_0.73_b_1.90_Sigma_0.05.pth"

evaluate_score_convergence(
model_path = model_path,
kernel_path = kernel_path,
path_path = path_path,
burn_in=500,         
max_plot=999,
hidden_dim =128,
num_layers=4,
device="cuda"
)

MSE        = 9.581265e-02
VarScale   = 1.447963e+00
Rel. Error = 6.617063e-02


In [33]:
# ============================
# Model and data paths
# ============================
d = 10
hidden_dim = 128
num_layers = 4

# Load the tensor
hybrid_path = "Gaussian_change detection/hybrid_path_P0toP1.pt" 
data = torch.load(hybrid_path, map_location="cpu")



# P_0
path_path_P0 = "Gaussian_change detection/P_0_messy_markov_path_A_0.73_b_1.90_Sigma_0.05.pt"
#  model and kernel
kernel_path_P0 = "Gaussian_change detection/P_0_messy_kernel_params_A_0.73_b_1.90_Sigma_0.05.pt"
model_path_P0 = "Gaussian_change detection/P_0_128_4_messy_markov_model_A_0.73_b_1.90_Sigma_0.05.pth"


# P_1
path_path_P1 = "Gaussian_change detection/P_1_messy_markov_path_A_0.66_b_1.90_Sigma_0.09.pt"
kernel_path_P1 = "Gaussian_change detection/P_1_messy_kernel_params_A_0.66_b_1.90_Sigma_0.09.pt"
model_path_P1 = "Gaussian_change detection/P_1_128_4_messy_markov_model_A_0.66_b_1.90_Sigma_0.09.pth"

# ============================
# Load models and print detail
# ============================
model_P0 = ConditionalScoreNet(d, hidden_dim, num_layers).to("cuda")
model_P1 = ConditionalScoreNet(d, hidden_dim, num_layers).to("cuda")

model_P0.load_state_dict(torch.load(model_path_P0, map_location="cuda"))
model_P1.load_state_dict(torch.load(model_path_P1, map_location="cuda"))

model_P0.eval()
model_P1.eval()

print("===== Model P0 Architecture =====")
print(model_P0)
print("===== Evaluate score convergence =====")
# ============================
# Evaluate score convergence
# ============================
results_P0 = evaluate_score_convergence(
    model_path=model_path_P0,
    kernel_path=kernel_path_P0,
    path_path=hybrid_path,   # using same path for comparison
    burn_in=500,
    max_plot=999,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    device="cuda"
)

===== Model P0 Architecture =====
ConditionalScoreNet(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): SiLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): SiLU()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): SiLU()
    (8): Linear(in_features=128, out_features=10, bias=True)
  )
)
===== Evaluate score convergence =====
MSE        = 4.224834e+04
VarScale   = 6.035652e+04
Rel. Error = 6.999797e-01


In [30]:
print("\n===== Model P1 Architecture =====")
print(model_P1)
print("===== Evaluate score convergence =====")
# ============================
# Evaluate score convergence
# ============================
results_P1 = evaluate_score_convergence(
    model_path=model_path_P1,
    kernel_path=kernel_path_P1,
    path_path=hybrid_path,
    burn_in=500,
    max_plot=999,
    hidden_dim=hidden_dim,
    num_layers=num_layers,
    device="cuda"
)


===== Model P1 Architecture =====
ConditionalScoreNet(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): SiLU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): SiLU()
    (6): Linear(in_features=128, out_features=128, bias=True)
    (7): SiLU()
    (8): Linear(in_features=128, out_features=10, bias=True)
  )
)
===== Evaluate score convergence =====
MSE        = 9.581265e-02
VarScale   = 1.447963e+00
Rel. Error = 6.617063e-02
